# 线性回归正常实现

In [1]:
# 数据集跟 02 一样手动创建
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

与 02 不同，读取数据集使用框架中现有的 API。其中 `is_train` 表示是否希望数据迭代器在每个迭代周期内打乱数据

In [2]:
def load_array(data_arrays, batch_size, is_train=True):  #@save
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

next(iter(data_iter))

[tensor([[-1.5471,  0.2390],
         [-1.8552,  1.0031],
         [-1.5005, -0.4342],
         [ 0.2760,  0.6419],
         [ 0.6376,  1.4837],
         [-0.0766, -0.4384],
         [-0.0036,  0.1245],
         [-1.5263,  0.7978],
         [ 0.7752, -0.0042],
         [ 0.1970, -1.9491]]),
 tensor([[ 0.2888],
         [-2.9075],
         [ 2.6922],
         [ 2.5655],
         [ 0.4432],
         [ 5.5213],
         [ 3.7759],
         [-1.5577],
         [ 5.7650],
         [11.2115]])]

## 定义模型
首先定义一个模型变量`net`，他是一个`Sequential`类的实例。`Sequential`类将多个层串联在一起。当给定输入数据时，他的实例将数据传入第一层，然后将第一层输出作为第二层输入......

在 PyTorch 中全连接层在`Linear`类中定义

In [3]:
# nn是神经网络的缩写
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))

In [4]:
#初始化参数
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

#损失函数
loss = nn.MSELoss()

#优化算法
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

## 训练
在每个迭代周期里，我们将完整遍历一次数据集（`train_data`），
* 通过调用`net(X)`生成预测并计算损失`l`（前向传播）。
* 通过进行反向传播来计算梯度。
* 通过调用优化器来更新模型参数。

In [5]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.000161
epoch 2, loss 0.000102
epoch 3, loss 0.000101


In [6]:
#实际的误差
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)

w的估计误差： tensor([-0.0002, -0.0001])
b的估计误差： tensor([0.0005])
